# Solutions — Styling & Tailwind CSS

Only look here after you've actually tried the exercises in `styling_tailwind.ipynb`.

LESSON 11 and 12 are done in `react-scratch`, so those answers are written out. LESSON 13
runs here.

### LESSON 11 — Exercise

**3. Which file won.** Whichever CSS was imported **last** wins, because both rules have the
same specificity (`.box`) and the later one overrides the earlier. Change the order of the
two imports in `App.jsx` and the winner changes. DevTools shows this directly: the losing
rule appears struck through in the Styles panel.

This is the entire problem with global CSS. Nothing is wrong with either file — they simply
both claimed the name `box`, and the bundler had to pick one.

**4 and 5. After converting `Panel` to a module.** The collision stops, because `Panel` is
no longer using a class called `box`. Build the project and look in `dist/assets/`:

```css
._box_12hfz_1 { border: 1px solid blue; }   /* from Panel.module.css */
.box          { border: 1px solid red;  }   /* still global, from card.css */
```

The module's class was renamed at build time, so nothing else can match it. `Card` kept the
plain `.box` and is still exposed to any other file that wants that name.

**The thing to take away:** CSS Modules do not make CSS "scoped" by magic. They rename your
class so that nobody else can collide with it — a mechanical fix to a naming problem.

### LESSON 11 — Mini challenge

1. **`box-sizing` for every element — global CSS.** It is a page-wide reset; there is no
   component to scope it to.
2. **Card's radius and padding — CSS Module.** It belongs to one component, and a generated
   name means nobody else's `.card` can interfere.
3. **Chart bar height from data — inline `style`.** The value is not known until the app
   runs. This is exactly the case React's docs reserve `style` for.
4. **Two unrelated `.title` classes — CSS Modules.** This is the collision from the
   exercise. Each component gets its own generated name, and both can keep the obvious name.
5. **The brand colour in eleven components — global CSS, as a custom property.**

```css
:root { --brand: #2563eb; }
```

Then every module refers to `var(--brand)`.

**Why the two tempting answers are wrong.** Inline `style` means the colour is now written
in eleven JavaScript files, and changing it is eleven edits — with the one you miss shipping
to production. Copying the hex into eleven CSS Modules has the same problem and is harder to
grep, because a rebrand touches every file.

A value used in many places wants **one definition and many references**. That is what a
custom property gives you, and it is also, in a different form, what Tailwind's colour scale
gives you.

### LESSON 12 — Exercise

The layout the exercise asks for:

```jsx
export default function App() {
  return (
    <div className="min-h-screen bg-slate-50 p-6">
      <header className="flex items-center justify-between">
        <div>
          <h1 className="text-2xl font-bold text-slate-900">Dashboard</h1>
          <p className="text-sm text-slate-500">Everything at a glance</p>
        </div>
        <button className="rounded-lg bg-blue-600 px-4 py-2 font-medium text-white">
          Refresh
        </button>
      </header>

      <div className="mt-6 grid grid-cols-1 gap-4 md:grid-cols-3">
        <div className="rounded-lg border border-slate-200 bg-white p-4 hover:border-slate-400">
          <p className="text-sm text-slate-500">Revenue</p>
          <p className="mt-1 text-xl font-bold text-slate-900">12,400 EUR</p>
        </div>
        {/* two more like it */}
      </div>
    </div>
  );
}
```

**Notes on the pieces.**

- `justify-between` does the work in the header: the title block goes left, the button goes
  right, and no margins or floats are involved. `items-center` keeps them aligned vertically.
- `grid-cols-1 md:grid-cols-3` is the mobile-first pattern. One column is the default; three
  columns take over from the `md` breakpoint. Writing `grid-cols-3 md:grid-cols-1` would be
  backwards — desktop first, then a narrower override.
- The breakpoint you find by narrowing the window is `48rem`, which is 768px at the default
  font size.
- The three cards repeat, identically, and you cannot yet do anything about it. That is the
  mini-project's whole subject.

### LESSON 12 — Mini challenge

**1. The claim holds.** A class you used is in `dist/assets/*.css`; `bg-fuchsia-700` is not
there at all. Tailwind emits CSS only for what it found in your source.

**2. Why the interpolated class fails.** Tailwind reads your **source file as text**, before
anything runs. In the file it finds the literal characters `text-blue-` followed by a
`${...}` expression, and `text-blue-` is not a class name, so nothing is generated. The
variable `shade` only has the value `600` while the app is running, which is far too late —
the CSS was written at build time.

**3. Searching the built CSS.** `text-blue-600` is not there. The paragraph really does
carry `class="text-blue-600"` in the page, which is what makes this confusing to debug: the
class is on the element, it simply matches no rule.

For Tailwind to get this right it would have to execute your JavaScript to discover which
strings your code can produce — for every code path, with every possible value. Reading text
is a deliberate trade: it is fast and predictable, and the price is the rule in LESSON 13.

### LESSON 13 — Exercise

In [ ]:
function l13cx(...values) {
  return values.filter(Boolean).join(" ");
}

const l13variantClasses = {
  primary: "bg-blue-600 text-white",
  secondary: "bg-slate-200 text-slate-900",
  danger: "bg-red-600 text-white",
};

function l13buttonClasses(variant, isDisabled) {
  return l13cx(
    "px-4 py-2 rounded-lg font-medium",
    l13variantClasses[variant] ?? l13variantClasses.secondary,
    isDisabled && "opacity-50 cursor-not-allowed",
  );
}

console.log(l13buttonClasses("primary", false));
console.log(l13buttonClasses("danger", true));
console.log(l13buttonClasses("secondary", false));
console.log(l13buttonClasses("ghost", false));

// Why interpolating the variant into the class name is wrong, twice over:
// 1. Tailwind never sees a complete class name, so no rule is generated.
// 2. It does not even describe these variants: secondary is bg-slate-200, not -600,
//    and the text colour changes per variant too. The pattern is not a pattern.

**Common mistakes.**

- Using `||` instead of `??` for the fallback. Both work here, but `||` also replaces any
  falsy value — and if a variant's classes were ever an empty string, `||` would quietly
  substitute the default. `??` only catches `null` and `undefined`, which is what "unknown
  variant" actually means.
- Leaving out the fallback entirely. `l13variantClasses["ghost"]` is `undefined`, and
  without `filter(Boolean)` you would get the literal text `undefined` inside `className`.
  Try it once: it is a bug that looks bizarre in DevTools and takes longer to find than it
  should.
- `values.join(" ")` without the filter. A `false` condition renders as `"false"` in the
  class attribute.

### LESSON 13 — Mini challenge

**Part 1.**

| # | snippet | generated? | why |
|---|---|---|---|
| 1 | an interpolated `` `p-${size}` `` | **no** | the source contains `p-`, which is not a class name |
| 2 | `size === "lg" ? "p-6" : "p-2"` | **yes, both** | both names appear complete in the file |
| 3 | `const classes = "p-6"` | **yes** | a complete name in the source; Tailwind does not care that it sits in a variable |
| 4 | `{ lg: "p-6", sm: "p-2" }` | **yes, both** | both values are complete strings, and both are generated even if only one is ever used |

The pattern across 2, 3 and 4: it does not matter where the string lives, or whether it is
ever reached. It matters only that the characters are there to be found.

**Part 2 — the comment trick.** It works for exactly the reason the rest of this lesson
works: Tailwind scans the file as text and has no idea what a comment is. The characters
`bg-fuchsia-700` are in the file, so the CSS gets generated.

Two reasons not to ship it:

1. **It is invisible glue.** Nothing connects that comment to the code relying on it. The
   next person tidies up a stale comment, the CSS silently stops being generated, and the
   bug surfaces somewhere else entirely with no clue pointing back.
2. **Nothing keeps it in sync.** Add a fourth fuchsia shade and it breaks again, with no
   error — just an element wearing a class that matches no rule.

**What they should do instead:** put the complete class names in a lookup object, the way
Part 1's option 4 does. The strings are then in the source *because the code uses them*,
which is a connection that survives refactoring.